<a href="https://colab.research.google.com/github/AmanuelDaget/YOLOv12-Object-Recognition-from-Remote-Sensing-images/blob/main/YOLOv12_Object_Recognition_from_Remote_Sensing_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Object Recognition using YOLOv12 from Remote Sensing images**

**Install important libraries**

In [2]:
# !pip install ultralytics torchvision pyyaml opencv-python -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.8 MB/s eta 0:00:00


**Import libraries**

In [3]:
import cv2, shutil, random, collections
import numpy as np
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


**MOUNT GOOGLE DRIVE**

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


**CONFIG**

In [5]:
ROOT     = '/content/drive/MyDrive/Colab Notebooks/Computer Vision and Image Processing/Object-detection-using-YOLOv12'
OUT      = '/content/drive/MyDrive/Colab Notebooks/Computer Vision and Image Processing/Object-detection-using-YOLOv12/yolo_processed'
YAML     = f'{OUT}/simd.yaml'
RUNS     = '/content/drive/MyDrive/runs'
TILE     = 512
OVERLAP  = 0.2
MIN_VIS  = 0.3
RARE_CLS = {1, 7}   # update after checking class distribution
OVERSAMPLE = 3

**PREPROCESSING**

In [ ]:
def apply_clahe(image_path):
    """ Apply CLAHE to improve local contrast. Very useful for satellite imagery. """

    img = cv2.imread(str(image_path))

    if img is None:
        return

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    cl = clahe.apply(l)

    merged = cv2.merge((cl, a, b))

    enhanced = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

    cv2.imwrite(str(image_path), enhanced)



**PREPARE DATASET**

In [ ]:
def prepare_dataset(val_ratio=0.10, seed=42):

    print("\n[INFO] Preparing Dataset...")

    train_img_dir = SIMD_ROOT / "training" / "images"
    train_lbl_dir = SIMD_ROOT / "training" / "labels"

    val_img_dir = SIMD_ROOT / "val" / "images"
    val_lbl_dir = SIMD_ROOT / "val" / "labels"

    val_img_dir.mkdir(parents=True, exist_ok=True)
    val_lbl_dir.mkdir(parents=True, exist_ok=True)

    # ──────────────────────────────────────────────────────────────────────
    # CREATE VALIDATION SPLIT
    # ──────────────────────────────────────────────────────────────────────
    existing_val = list(val_img_dir.glob("*"))

    if len(existing_val) == 0:

        imgs = list(train_img_dir.glob("*.[Jj][Pp][Gg]"))

        random.seed(seed)
        random.shuffle(imgs)

        n_val = int(len(imgs) * val_ratio)

        val_imgs = imgs[:n_val]

        for img_path in val_imgs:

            shutil.move(
                str(img_path),
                str(val_img_dir / img_path.name)
            )

            lbl_name = img_path.with_suffix(".txt").name

            lbl_path = train_lbl_dir / lbl_name

            if lbl_path.exists():

                shutil.move(
                    str(lbl_path),
                    str(val_lbl_dir / lbl_name)
                )

        print(f"[INFO] Validation split created: {n_val} images")

    else:
        print("[INFO] Validation split already exists")

    # ──────────────────────────────────────────────────────────────────────
    # APPLY CLAHE
    # ──────────────────────────────────────────────────────────────────────
    if USE_CLAHE:

        print("\n[INFO] Applying CLAHE preprocessing...")

        all_dirs = [
            SIMD_ROOT / "train" / "images",
            SIMD_ROOT / "val" / "images",
            SIMD_ROOT / "test" / "images"
        ]

        for folder in all_dirs:

            imgs = list(folder.glob("*.[Jj][Pp][Gg]"))

            for i, img_path in enumerate(imgs):

                apply_clahe(img_path)

                if i % 500 == 0:
                    print(f"  Processed {i}/{len(imgs)}")

    # ──────────────────────────────────────────────────────────────────────
    # WRITE YAML
    # ──────────────────────────────────────────────────────────────────────
    yaml_data = {
        "path": str(SIMD_ROOT),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": NC,
        "names": CLASSES
    }

    with open("simd.yaml", "w") as f:
        yaml.dump(
            yaml_data,
            f,
            sort_keys=False
        )

    print("\n[INFO] simd.yaml created")

